In [ ]:
import pandas as pd
from itertools import product

excel_file = "resultados.xlsx"

architectures = [
    [32], [64], [128], [264],
    [8, 4], [16, 8], [32, 16], [64, 32], [128, 64], [264, 128],
    [16, 8, 4], [32, 16, 8], [128, 64, 32], [264, 128, 64]
]

r_values = [0.01, 0.5, 0.9]

N_MODELS = 5  # número de seeds

df = pd.read_excel(excel_file)

comb = list(product(architectures, r_values))

for i, (arch, r) in enumerate(comb):
    
    start = i * N_MODELS
    end = start + N_MODELS
    
    df.loc[start:end-1, "Neurons"] = str(arch)
    df.loc[start:end-1, "r"] = r

df.to_excel("resultados.xlsx", index=False)

In [ ]:
import numpy as np

TARGETS = ["Theta"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [19]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [20]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
display(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====


,Target,Dataset,Best_Model,Neurons,Best_R2
0,Theta,Train_1,model_128_4,"[264, 128]",[0.9411809843478175]
1,Theta,Train_2,model_64_0,"[128, 64]",[0.8604988053965039]
2,Theta,Val,model_128_3,"[264, 128]",[0.7682549582681082]
3,Theta,Test_1,model_264_0,[264],[0.8038970319523201]
4,Theta,Test_2,model_4_1,"[8, 4]",[0.9335510940067954]
5,Theta,Test_3,model_8_0,"[16, 8]",[0.8388578406059151]


In [22]:
df_summary_top10["bestR2"] = best_only_df["Best_R2"].str.strip("[]").astype(float)
df_summary_top10["Neurons"] = best_only_df["Neurons"]
display(df_summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,Theta,0.894167,0.042872,0.024158,0.009786,0.941181,"[264, 128]"
1,Train_2,Theta,0.785575,0.036818,0.048954,0.008406,0.860499,"[128, 64]"
2,Val,Theta,0.404839,0.272290,0.194784,0.089115,0.768255,"[264, 128]"
3,Test_1,Theta,0.691886,0.072697,0.070780,0.016700,0.803897,[264]
4,Test_2,Theta,0.782787,0.070042,0.051515,0.016611,0.933551,"[8, 4]"
5,Test_3,Theta,0.593067,0.119857,0.133778,0.039403,0.838858,"[16, 8]"


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted